# 14 — Nightly pipeline validation

**Project:** Hotel overbooking decision support  
**Owner:** Build Lead  
**Purpose:** Inspect and validate the inference-only nightly recommendation prototype.

This notebook audits the actual implementation in `src/nightly.py`, the frozen model artifacts, the tests and the run manifest. It focuses on whether the pipeline produces a correctly shaped recommendation table and whether the output is safe to interpret.

**Important:** A successful historical smoke test is not proof of production readiness. The historical CSV is not an as-of booking snapshot, and the project does not have confirmed room inventory or observed guest-walk records.

In [1]:
# Run from the repository root, so `from src import ...` and the relative evidence/ paths work
# whether this notebook is opened from the repo folder or from notebooks/pipeline/.
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "contract.py").exists())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", REPO_ROOT)

Working directory: C:\Users\Lenovo\Desktop\Build_2_Repo\IIT_GN_W06


## 1. Understand the pipeline contract

The command-line prototype is designed to:
- load a booking extract;
- load a frozen XGBoost pipeline and conformal calibration state;
- generate booking-level cancellation probabilities;
- aggregate those probabilities to hotel × arrival date;
- produce a 14-night recommendation table;
- save a CSV and a JSON manifest.

It performs inference only. It does not fit, tune or recalibrate the model.

The example command in the implementation is:

```bash
python src/nightly.py --input data/hotels.csv --from 2017-08-18 --nights 14
```

The default model artifacts are:
- `models/xgboost.joblib`
- `models/xgboost_conformal_state.joblib`
- `models/conformal_quantiles.json`

## 2. Check required project files

This cell reports what is actually present in the current working tree. Missing artifacts are not silently recreated.

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

required_paths = [
    "src/nightly.py",
    "src/features.py",
    "src/contract.py",
    "tests/test_nightly.py",
    "models/xgboost.joblib",
    "models/xgboost_conformal_state.joblib",
    "models/conformal_quantiles.json",
    "data/hotels.csv"
]

file_check = pd.DataFrame([
    {
        "path": name,
        "exists": Path(name).exists(),
        "bytes": Path(name).stat().st_size if Path(name).exists() else None
    }
    for name in required_paths
])

display(file_check)

,path,exists,bytes
0,src/nightly.py,True,12118
1,src/features.py,True,2674
2,src/contract.py,True,5131
3,tests/test_nightly.py,True,1488
4,models/xgboost.joblib,True,116786
5,models/xgboost_conformal_state.joblib,True,9209
6,models/conformal_quantiles.json,True,2245
7,data/hotels.csv,True,16974990


## 3. Read the implementation rather than relying on a summary

The next cell displays the source code so that the implementation can be reviewed directly. Pay particular attention to:
- the input requirements;
- how arrival dates are parsed;
- which columns are passed into the model;
- how missing hotel-nights are represented;
- how the oversell count is calculated;
- what the run manifest records.

In [3]:
source_path = Path("src/nightly.py")

if source_path.exists():
    source_text = source_path.read_text(encoding="utf-8")
    print(source_text)
else:
    print("src/nightly.py is not present in this working tree.")

"""Frozen-model nightly cancellation and overbooking recommendation pipeline.

Example:
    python src/nightly.py --input data/hotels.csv --from 2017-08-18 --nights 14

This script performs inference only. It never fits, tunes, or recalibrates a model.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import math
import platform
import sys
import uuid
from datetime import date
from pathlib import Path
from typing import Iterable

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(Path(__file__).resolve().parent))
from features import CATEGORICAL, FEATURE_COLUMNS, NUMERIC, make_features  # noqa: E402

EXPECTED_HOTELS = ("City Hotel", "Resort Hotel")
TARGET = "is_canceled"
DATE_PARTS = ("arrival_date_year", "arrival_date_month", "arrival_date_day_of_month")
NIGHT_PARTS = ("stays_in_weekend_nights", "stays_in_week_nights")


def sha256(path: Path) -> str:
   

## 4. Validate the prediction feature schema

The source implementation expects a fitted scikit-learn pipeline with `prep` and `model` steps. Its categorical and numeric columns must match the saved feature schema.

A known project-level issue must be checked before running the model: the earlier feature-selection notebook held `country` back pending timing validation, while the existing `src/features.py` and nightly model schema include `country`.

Do not ignore this mismatch. Decide and document which feature schema is approved, then make the notebook, `src/features.py`, saved model and nightly schema agree.

In [4]:
features_path = Path("src/features.py")

if features_path.exists():
    print(features_path.read_text(encoding="utf-8"))
else:
    print("src/features.py is not present.")

"""Conservative candidate feature extraction for the hotel cancellation project.

This module deliberately uses only the candidate first-bin columns from docs/COLUMNS.md.
It does not perform model fitting, target encoding, or imputation of children.
The source CSV is a near-arrival snapshot rather than a guaranteed original-booking snapshot;
validate the fields against the PMS change log before production use.
"""
from __future__ import annotations

import pandas as pd

TARGET = "is_canceled"

# Approved candidate fields (notebooks/pipeline/05_feature_engineering.ipynb, docs/DECISIONS.md P3).
# Revisit after verifying original-booking timestamps.
CATEGORICAL = (
    "hotel",
    "market_segment",
    "distribution_channel",
    "agent",
    "company",
)
NUMERIC = (
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
)
FEATURE_COLUMNS = CATEGORICAL + NUMERIC

# Held back until booking-time availability is proven: the source paper warns that the c

## 5. Review the date-window and active-booking logic

The prototype creates a requested date grid for both hotels and the requested number of nights. It aggregates whatever rows are present in the input extract for those dates.

**Critical operational question:** Does the input contain only bookings that are active and known as of the recommendation timestamp?

The current implementation does not accept an `as_of` timestamp and does not filter an active-booking population using a booking-status snapshot. Passing the full historical CSV means the input includes bookings that were later cancelled, and possibly records that would not have been known at the decision time.

Therefore, a historical full-file smoke test verifies code execution and output shape, not the validity of a live booking population.

In [5]:
# Inspect the input loader and report builder directly.
if source_path.exists():
    lines = source_text.splitlines()
    keywords = [
        "def load_bookings",
        "def validate_model_schema",
        "def build_report",
        "def run(args",
        "def parse_args"
    ]
    for keyword in keywords:
        matches = [i for i, line in enumerate(lines) if keyword in line]
        print(f"\n{keyword}")
        if matches:
            start = matches[0]
            print("\n".join(
                f"{i+1}: {lines[i]}"
                for i in range(start, min(start + 24, len(lines)))
            ))
        else:
            print("Not found")


def load_bookings
45: def load_bookings(path: Path) -> pd.DataFrame:
46:     if not path.exists():
47:         raise FileNotFoundError(f"Input file does not exist: {path}")
48:     df = pd.read_csv(path)
49:     if "arrival_date" not in df.columns:
50:         missing = [c for c in DATE_PARTS if c not in df.columns]
51:         if missing:
52:             raise ValueError(f"Input needs arrival_date or all date parts; missing: {missing}")
53:         df["arrival_date"] = pd.to_datetime(
54:             df["arrival_date_year"].astype(str)
55:             + "-" + df["arrival_date_month"].astype(str)
56:             + "-" + df["arrival_date_day_of_month"].astype(str),
57:             format="%Y-%B-%d",
58:             errors="coerce",
59:         )
60:     else:
61:         df["arrival_date"] = pd.to_datetime(df["arrival_date"], errors="coerce")
62: 
63:     if df["arrival_date"].isna().any():
64:         bad = int(df["arrival_date"].isna().sum())
65:         raise ValueError(f"{bad} rows

## 6. Test the nightly aggregation with a small synthetic example

This test is intentionally synthetic. It checks that:
- booking probabilities are summed by hotel and arrival date;
- the requested grid includes dates with no input bookings;
- missing hotel-nights are flagged instead of receiving a fabricated recommendation;
- the policy output is a whole-number oversell count where data exists.

It does not test the predictive accuracy of XGBoost.

In [6]:
import sys
import numpy as np
import pandas as pd

src_dir = str(Path("src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

try:
    from nightly import build_report, one_sided_lower_quantile

    synthetic_bookings = pd.DataFrame({
        "hotel": ["City Hotel", "City Hotel", "Resort Hotel"],
        "arrival_date": pd.to_datetime([
            "2017-08-18", "2017-08-18", "2017-08-19"
        ])
    })

    synthetic_probabilities = np.array([0.20, 0.30, 0.60])

    synthetic_report = build_report(
        synthetic_bookings,
        synthetic_probabilities,
        start=pd.Timestamp("2017-08-18"),
        nights=2,
        model_name="SYNTHETIC_TEST_ONLY",
        q95_lower=-2.0,
        q95_upper=3.0,
        q99_lower_residual=-1.0
    )

    display(synthetic_report)

    assert len(synthetic_report) == 4
    city_night = synthetic_report[
        (synthetic_report["hotel"] == "City Hotel")
        & (synthetic_report["arrival_date"] == pd.Timestamp("2017-08-18"))
    ].iloc[0]

    assert city_night["bookings"] == 2
    assert abs(city_night["expected_cancellations"] - 0.50) < 1e-9
    assert city_night["recommended_oversell"] == 0

    missing_night = synthetic_report[
        (synthetic_report["hotel"] == "City Hotel")
        & (synthetic_report["arrival_date"] == pd.Timestamp("2017-08-19"))
    ].iloc[0]

    assert missing_night["status"] == "NO_BOOKINGS_DATA"
    assert pd.isna(missing_night["recommended_oversell"])

    print("Synthetic aggregation checks passed.")
except Exception as error:
    print("Synthetic test could not run:", repr(error))
    print("Check dependencies and the source module before proceeding.")

,hotel,arrival_date,bookings,expected_cancellations,probability_spread,interval_95_lower,interval_95_upper,lower_99_policy_bound,recommended_oversell,model,status
0,City Hotel,2017-08-18,2,0.5,0.608276,0.0,2.0,0.0,0,SYNTHETIC_TEST_ONLY,OK
1,City Hotel,2017-08-19,0,NaN,NaN,NaN,NaN,NaN,<NA>,SYNTHETIC_TEST_ONLY,NO_BOOKINGS_DATA
2,Resort Hotel,2017-08-18,0,NaN,NaN,NaN,NaN,NaN,<NA>,SYNTHETIC_TEST_ONLY,NO_BOOKINGS_DATA
3,Resort Hotel,2017-08-19,1,0.6,0.489898,0.0,1.0,0.0,0,SYNTHETIC_TEST_ONLY,OK


Synthetic aggregation checks passed.


## 7. Check the finite-sample conformal quantile calculation

The implementation uses the lower-tail finite-sample rank:

\[
r = \lceil (n+1)\alpha \rceil
\]

For 148 calibration residuals and alpha 0.01, the expected rank is 2. This cell checks the function against that stated rule.

In [7]:
try:
    q_value, rank = one_sided_lower_quantile(range(148), alpha=0.01)
    print("Returned rank:", rank)
    print("Returned quantile for test sequence:", q_value)
    assert rank == 2
    assert q_value == 1.0
    print("Finite-sample rank check passed.")
except Exception as error:
    print("Quantile check could not run:", repr(error))

Returned rank: 2
Returned quantile for test sequence: 1.0
Finite-sample rank check passed.


## 8. Inspect the saved smoke-test evidence

The implementation audit document reports a historical smoke test for 18–31 August 2017:
- 119,390 input booking rows;
- 28 report rows (14 dates × 2 hotels);
- all 28 hotel-nights flagged `OK`;
- no model fitting, tuning or recalibration during inference.

The following cell checks whether the output and manifest exist in this working tree. It does not assume they are present.

In [8]:
output_path = Path("outputs/nightly_recommendations.csv")
manifest_path = Path("outputs/nightly_recommendations.manifest.json")

print("Output exists:", output_path.exists())
print("Manifest exists:", manifest_path.exists())

if output_path.exists():
    historical_output = pd.read_csv(output_path)
    display(historical_output.head(10))
    print("Output rows:", len(historical_output))
    print("Unique hotels:", historical_output["hotel"].nunique())
    print("Unique dates:", historical_output["arrival_date"].nunique())
    if "status" in historical_output:
        print("Status counts:")
        display(historical_output["status"].value_counts(dropna=False))

if manifest_path.exists():
    import json
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([
        {"field": key, "value": value}
        for key, value in manifest.items()
    ]))

Output exists: True
Manifest exists: True


,hotel,arrival_date,bookings,expected_cancellations,probability_spread,interval_95_lower,interval_95_upper,lower_99_policy_bound,recommended_oversell,model,status
0,City Hotel,2017-08-18,121,26.694475,4.359274,17.178384,93.200510,13.171369,13,XGBoost,OK
1,City Hotel,2017-08-19,122,38.060267,4.497694,28.544176,104.566302,24.537161,24,XGBoost,OK
2,City Hotel,2017-08-20,101,18.114181,3.595198,8.598091,84.620216,4.591076,4,XGBoost,OK
3,City Hotel,2017-08-21,94,18.681340,3.657038,9.165250,85.187375,5.158235,5,XGBoost,OK
4,City Hotel,2017-08-22,60,10.946611,2.846750,1.430521,60.000000,0.000000,0,XGBoost,OK
5,City Hotel,2017-08-23,86,15.503797,3.344524,5.987707,82.009832,1.980692,1,XGBoost,OK
6,City Hotel,2017-08-24,91,19.340844,3.667385,9.824753,85.846878,5.817738,5,XGBoost,OK
7,City Hotel,2017-08-25,139,20.433775,3.723056,10.917684,86.939810,6.910669,6,XGBoost,OK
8,City Hotel,2017-08-26,101,20.540918,3.829911,11.024828,87.046953,7.017813,7,XGBoost,OK
9,City Hotel,2017-08-27,107,24.312940,4.130344,14.796850,90.818975,10.789834,10,XGBoost,OK


Output rows: 28
Unique hotels: 2
Unique dates: 14
Status counts:


status
OK    28
Name: count, dtype: int64

,field,value
0,run_id,a4e682e2-4c26-4dd7-b76b-af84fba4c418
1,status,SUCCESS
2,run_timestamp_utc,2026-11-15T07:26:17.376787+00:00
3,input_path,/mnt/data/IIT_GN_W06_extracted/IIT_GN_W06/data...
4,input_sha256,d10cdeb07de9309007f528d6499a2c0467ae7a6b287b80...
5,input_rows,119390
6,rows_scored,119390
7,requested_from,2017-08-18
8,requested_nights,14
9,report_rows,28


## 9. Optional: run the frozen pipeline if all required artifacts exist

This is an actual end-to-end execution attempt using the historical CSV and the documented 14-night example. It is optional because model artifacts or dependencies may not be installed in every checkout.

**Interpretation warning:** even if this command succeeds, it is a historical smoke test, not a valid live forecast. It uses a retrospective dataset and cannot establish that only bookings known at the time were scored.

In [9]:
import subprocess
import sys

pipeline_inputs = [
    Path("src/nightly.py"),
    Path("data/hotels.csv"),
    Path("models/xgboost.joblib"),
    Path("models/xgboost_conformal_state.joblib"),
    Path("models/conformal_quantiles.json")
]

can_attempt_pipeline = all(path.exists() for path in pipeline_inputs)

if can_attempt_pipeline:
    command = [
        sys.executable, "src/nightly.py",
        "--input", "data/hotels.csv",
        "--from", "2017-08-18",
        "--nights", "14",
        "--output", "outputs/notebook14_smoke_test.csv"
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False
    )

    print("Command:", " ".join(command))
    print("Return code:", result.returncode)
    print("\nSTDOUT:\n", result.stdout)
    print("\nSTDERR:\n", result.stderr)
else:
    print("Skipped: one or more required model/input artifacts are missing.")
    print("Missing:", [str(path) for path in pipeline_inputs if not path.exists()])

Command: C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\python.exe src/nightly.py --input data/hotels.csv --from 2017-08-18 --nights 14 --output outputs/notebook14_smoke_test.csv
Return code: 1

STDOUT:
 

STDERR:
 C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/m

## 10. Operational limitations and production blockers

The current prototype has several limitations that must remain visible:

| Area | Current status | Why it matters |
|---|---|---|
| As-of snapshot | Not implemented | The model must score only bookings known and active at the decision time |
| Active booking filter | Not implemented | Historical future outcomes must not leak into the operational population |
| Feature timing | Not fully verified | A column in the CSV is not necessarily known at booking or decision time |
| Room capacity | Not available | A recommended oversell count cannot be checked against physical inventory |
| Booking-to-room mapping | Not verified | One booking row is only a room-equivalent under the contract |
| Actual walks | Not available | Simulated walked guests are not observed relocations |
| Group walk ceiling | Not guaranteed by per-night interval | The aggregate guardrail needs a separate evaluation |
| Missing hotel-night | Flagged | No recommendation should be fabricated when there is no input data |

The pipeline is an inference prototype, not an approved production booking decision system.

## 11. Write a validation checklist for the build review

Complete the status column after you have run the notebook and reviewed the source. Do not mark a test as passed merely because the code cell exists.

In [10]:
validation_checklist = pd.DataFrame([
    ["Source file inspected", "Review src/nightly.py and understand each stage", "Pending"],
    ["Feature schema reconciled", "Notebook 05, src/features.py and saved model agree", "Pending"],
    ["Synthetic aggregation test", "Expected cancellation sum and missing-night flag verified", "Pending"],
    ["Conformal rank test", "Finite-sample rank matches the documented formula", "Pending"],
    ["Historical smoke test", "Run command and inspect output plus manifest", "Pending"],
    ["As-of snapshot", "Input contains only bookings known and active at decision time", "Not implemented"],
    ["Capacity integration", "Physical room inventory and room-equivalent mapping verified", "Not available"],
    ["Walk validation", "Compare predictions with actual relocation/walk records", "Not available"]
], columns=["check", "evidence_required", "status"])

display(validation_checklist)

,check,evidence_required,status
0,Source file inspected,Review src/nightly.py and understand each stage,Pending
1,Feature schema reconciled,"Notebook 05, src/features.py and saved model a...",Pending
2,Synthetic aggregation test,Expected cancellation sum and missing-night fl...,Pending
3,Conformal rank test,Finite-sample rank matches the documented formula,Pending
4,Historical smoke test,Run command and inspect output plus manifest,Pending
5,As-of snapshot,Input contains only bookings known and active ...,Not implemented
6,Capacity integration,Physical room inventory and room-equivalent ma...,Not available
7,Walk validation,Compare predictions with actual relocation/wal...,Not available


In [11]:
# Record what this run actually established. Every status below is computed here, not typed in.
import importlib
import json
import warnings

import joblib

evidence_folder = Path("evidence/nightly_pipeline_validation")
evidence_folder.mkdir(parents=True, exist_ok=True)
import features
importlib.reload(features)


def check(function):
    try:
        function()
        return "Passed"
    except Exception as error:  # record the failure rather than hide it
        return f"Failed: {error!r}"[:300]


def synthetic_check():
    report = build_report(synthetic_bookings, synthetic_probabilities, start=pd.Timestamp("2017-08-18"), nights=2,
                          model_name="SYNTHETIC_TEST_ONLY", q95_lower=-2.0, q95_upper=3.0, q99_lower_residual=-1.0)
    city = report[(report["hotel"] == "City Hotel") & (report["arrival_date"] == pd.Timestamp("2017-08-18"))].iloc[0]
    missing = report[(report["hotel"] == "City Hotel") & (report["arrival_date"] == pd.Timestamp("2017-08-19"))].iloc[0]
    assert len(report) == 4 and city["bookings"] == 2 and abs(city["expected_cancellations"] - 0.5) < 1e-9
    assert missing["status"] == "NO_BOOKINGS_DATA" and pd.isna(missing["recommended_oversell"])


def rank_check():
    assert one_sided_lower_quantile(range(148), alpha=0.01) == (1.0, 2)


with warnings.catch_warnings(record=True) as load_warnings:
    warnings.simplefilter("always")
    artifact = joblib.load("models/xgboost.joblib")
artifact_schema = {name: list(columns) for name, _, columns in artifact.named_steps["prep"].transformers}
approved_schema = {"cat": list(features.CATEGORICAL), "num": list(features.NUMERIC)}
artifact_matches = artifact_schema.get("cat") == approved_schema["cat"] and artifact_schema.get("num") == approved_schema["num"]

smoke_ran = can_attempt_pipeline
import re

stderr_plain = re.sub(r"\[[0-9;]*m", "", result.stderr) if smoke_ran else ""  # drop terminal colour codes
smoke_status = ("Not attempted: artifacts missing" if not smoke_ran
                else "Passed" if result.returncode == 0
                else f"Failed (exit code {result.returncode}): " + stderr_plain.strip().splitlines()[-1][:300])

actual_checklist = pd.DataFrame([
    ["Source file inspected", "Displayed and reviewed in sections 3 to 5 of this notebook", "Done"],
    ["Feature schema reconciled (code)", "src/features.py matches Notebook 05; tests/test_features.py guards it",
     "Done" if "country" not in features.FEATURE_COLUMNS else "Not done"],
    ["Feature schema reconciled (saved model)", f"models/xgboost.joblib schema {artifact_schema}",
     "Matches approved schema" if artifact_matches else "MISMATCH: artifact uses held-back `country`; must be retrained"],
    ["Synthetic aggregation test", "Expected cancellation sum and missing-night flag", check(synthetic_check)],
    ["Conformal rank test", "Rank 2 and value 1.0 for 148 residuals at alpha 0.01", check(rank_check)],
    ["Historical smoke test", "python src/nightly.py --input data/hotels.csv --from 2017-08-18 --nights 14", smoke_status],
    ["Artifact library versions", "Pickled with scikit-learn 1.8.0 / XGBoost 3.1.3; loaded with "
     f"scikit-learn {__import__('sklearn').__version__} / XGBoost {__import__('xgboost').__version__}",
     f"{len(load_warnings)} version warning(s) on load"],
    ["As-of snapshot", "Input contains only bookings known and active at decision time", "Not implemented"],
    ["Capacity integration", "Physical room inventory and room-equivalent mapping verified", "Not available"],
    ["Walk validation", "Compare predictions with actual relocation/walk records", "Not available"],
], columns=["check", "evidence", "status"])
display(actual_checklist)

actual_checklist.to_csv(evidence_folder / "validation_checklist.csv", index=False)
(evidence_folder / "smoke_test.json").write_text(json.dumps({
    "command": command if smoke_ran else None,
    "return_code": result.returncode if smoke_ran else None,
    "stderr_tail": stderr_plain.strip().splitlines()[-5:] if smoke_ran else None,
    "artifact_schema": artifact_schema,
    "approved_schema": approved_schema,
    "note": "A historical full-file run would prove code execution and output shape only, not live operational validity.",
}, indent=2), encoding="utf-8")
print("Saved validation evidence to", evidence_folder)

,check,evidence,status
0,Source file inspected,Displayed and reviewed in sections 3 to 5 of t...,Done
1,Feature schema reconciled (code),src/features.py matches Notebook 05; tests/tes...,Done
2,Feature schema reconciled (saved model),"models/xgboost.joblib schema {'cat': ['hotel',...",MISMATCH: artifact uses held-back `country`; m...
3,Synthetic aggregation test,Expected cancellation sum and missing-night flag,Passed
4,Conformal rank test,Rank 2 and value 1.0 for 148 residuals at alph...,Passed
5,Historical smoke test,python src/nightly.py --input data/hotels.csv ...,Failed (exit code 1): ValueError: Saved model ...
6,Artifact library versions,Pickled with scikit-learn 1.8.0 / XGBoost 3.1....,11 version warning(s) on load
7,As-of snapshot,Input contains only bookings known and active ...,Not implemented
8,Capacity integration,Physical room inventory and room-equivalent ma...,Not available
9,Walk validation,Compare predictions with actual relocation/wal...,Not available


Saved validation evidence to evidence\nightly_pipeline_validation


## 12. Final conclusion

The nightly pipeline demonstrates a useful engineering step: a frozen model can be loaded, probabilities can be aggregated to hotel-arrival nights, a policy can be applied, and a reproducible output manifest can be written.

It does **not** yet demonstrate live operational validity. The missing as-of active-booking snapshot, unresolved feature timing, absent room capacity and absent actual walk records are material blockers.

### Questions for your defence

1. Why is inference-only different from retraining?
2. Why does the pipeline aggregate booking probabilities to hotel-arrival nights?
3. Why does a missing hotel-night receive `NO_BOOKINGS_DATA` rather than an oversell recommendation?
4. What does the conformal lower bound represent, and what does it not guarantee?
5. Why is a full historical CSV unsuitable as a live as-of input?
6. What additional PMS data is required before deployment?
7. Why are simulated walked guests not the same as actual guests relocated to another hotel?

**End of the notebook sequence.** The next step is not another analytical notebook: reconcile the feature schema, run all notebooks in order in Codespaces, and update the project documentation with actual execution evidence and decisions.